In [1]:
DATASET_PATH = "ravdess"
AUDIO_PATH = f"{DATASET_PATH}/audio"
VIDEO_PATH = f"{DATASET_PATH}/video"
META_PATH = f"{DATASET_PATH}/metadata"


In [2]:
import os
import pandas as pd

emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

rows = []

for actor in sorted(os.listdir(AUDIO_PATH)):
    actor_dir = os.path.join(AUDIO_PATH, actor)
    if not os.path.isdir(actor_dir):
        continue

    for fname in sorted(os.listdir(actor_dir)):
        if not fname.endswith(".wav"):
            continue

        parts = fname.split("-")
        emotion_code = parts[2]
        emotion = emotion_map[emotion_code]

        audio_rel = os.path.join("audio", actor, fname)
        video_rel = os.path.join("video", actor, fname.replace(".wav", ".mp4"))

        rows.append({
            "actor": actor,
            "fname": fname,
            "audio_path": audio_rel,
            "video_path": video_rel,
            "emotion": emotion
        })

df = pd.DataFrame(rows)
os.makedirs(META_PATH, exist_ok=True)

labels_csv_path = os.path.join(META_PATH, "labels.csv")
df.to_csv(labels_csv_path, index=False)

labels_csv_path


'ravdess/metadata/labels.csv'

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os

class RavdessAudioDataset(Dataset):
    def __init__(self, csv_path, root_dir, sr=16000, n_mels=64, max_len=400):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.sr = sr
        self.n_mels = n_mels
        self.max_len = max_len

        # Map emotions to integers
        self.label2id = {label: i for i, label in enumerate(sorted(self.df["emotion"].unique()))}
        self.id2label = {v: k for k, v in self.label2id.items()}

    def __len__(self):
        return len(self.df)

    def _fix_length(self, mel_db):
        """
        mel_db: [n_mels, T]
        Make time dimension = self.max_len by padding or truncating.
        """
        n_mels, T = mel_db.shape

        if T < self.max_len:
            # pad at the end with zeros
            pad_width = self.max_len - T
            mel_db = np.pad(mel_db, ((0, 0), (0, pad_width)), mode="constant")
        else:
            # truncate (you can also center-crop if you like)
            mel_db = mel_db[:, :self.max_len]

        return mel_db

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.root_dir, row["audio_path"])

        # load audio
        y, sr = librosa.load(audio_path, sr=self.sr)

        # mel spectrogram
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=512,
            hop_length=160,
            n_mels=self.n_mels
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)  # [n_mels, T]

        # normalize
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        # fix length
        mel_db = self._fix_length(mel_db)  # [n_mels, max_len]

        # to tensor: [1, n_mels, max_len]
        x = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0)

        y_label = torch.tensor(self.label2id[row["emotion"]], dtype=torch.long)
        return x, y_label


In [4]:
csv_path = labels_csv_path  # from earlier
dataset = RavdessAudioDataset(csv_path, DATASET_PATH)

len(dataset), dataset[0][0].shape, dataset[0][1]

(1440, torch.Size([1, 64, 400]), tensor(5))

In [5]:
class AudioCNN(nn.Module):
    def __init__(self, n_mels=64, n_classes=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(64, n_classes)

    def forward(self, x):
        h = self.conv(x)          # [B, 64, 1, 1]
        h = h.view(h.size(0), -1) # [B, 64]
        out = self.fc(h)
        return out

n_classes = len(dataset.label2id)
model = AudioCNN(n_classes=n_classes).to("cuda" if torch.cuda.is_available() else "cpu")


In [6]:
from torch.utils.data import random_split, DataLoader

dataset_size = len(dataset)
val_size = int(0.2 * dataset_size)
train_size = dataset_size - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

len(train_dataset), len(val_dataset)

(1152, 288)

In [7]:
from tqdm.auto import tqdm
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 15  # more epochs now

for epoch in range(num_epochs):
    print(f"\n===== Epoch {epoch+1} / {num_epochs} =====")

    # ---- Train ----
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in tqdm(train_loader, desc=f"Train {epoch+1}", leave=False):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ---- Validate ----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X, y in tqdm(val_loader, desc=f"Val {epoch+1}", leave=False):
            X, y = X.to(device), y.to(device)

            logits = model(X)
            loss = criterion(logits, y)

            val_loss += loss.item() * X.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Train  Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val    Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")


===== Epoch 1 / 15 =====


Train 1:   0%|          | 0/72 [00:00<?, ?it/s]

Val 1:   0%|          | 0/18 [00:00<?, ?it/s]

Train  Loss: 2.0638 | Train Acc: 0.1337
Val    Loss: 2.0890 | Val   Acc: 0.1493

===== Epoch 2 / 15 =====


Train 2:   0%|          | 0/72 [00:00<?, ?it/s]

...

Val 14:   0%|          | 0/18 [00:00<?, ?it/s]

Train  Loss: 1.7095 | Train Acc: 0.3602
Val    Loss: 1.8325 | Val   Acc: 0.2639

===== Epoch 15 / 15 =====


Train 15:   0%|          | 0/72 [00:00<?, ?it/s]

Val 15:   0%|          | 0/18 [00:00<?, ?it/s]

Train  Loss: 1.6887 | Train Acc: 0.3507
Val    Loss: 1.8254 | Val   Acc: 0.2917


In [8]:
import torch

torch.save(model.state_dict(), "ravdess_audio_cnn.pth")
print("Saved model to ravdess_audio_cnn.pth")


Saved model to ravdess_audio_cnn.pth


In [9]:
import numpy as np
import pandas as pd
import torch
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

all_embeddings = []
all_rows = []

with torch.no_grad():
    for i in range(len(dataset)):
        # Get input + label
        x, y = dataset[i]          # x: [1, 64, max_len]
        x = x.unsqueeze(0).to(device)  # [1, 1, 64, max_len]

        # Forward until before final FC
        h = model.conv(x)          # [1, 64, 1, 1]
        h = h.view(h.size(0), -1)  # [1, 64]
        emb = h.squeeze(0).cpu().numpy()  # [64]

        # Metadata from the original df
        row = dataset.df.iloc[i]
        all_rows.append({
            "idx": i,
            "actor": row["actor"],
            "fname": row["fname"],
            "emotion": row["emotion"]
        })
        all_embeddings.append(emb)

# Convert to DataFrame
meta_df = pd.DataFrame(all_rows)
emb_dim = all_embeddings[0].shape[0]
emb_cols = [f"audio_emb_{j}" for j in range(emb_dim)]
emb_df = pd.DataFrame(all_embeddings, columns=emb_cols)

audio_features_df = pd.concat([meta_df, emb_df], axis=1)
audio_features_path = os.path.join(META_PATH, "audio_embeddings.csv")
audio_features_df.to_csv(audio_features_path, index=False)

audio_features_path, audio_features_df.head()

('ravdess/metadata/audio_embeddings.csv',
    idx    actor                     fname  emotion  audio_emb_0  audio_emb_1  \
 0    0  Actor_1  03-01-01-01-01-01-01.wav  neutral     0.091919     0.145806   
 1    1  Actor_1  03-01-01-01-01-02-01.wav  neutral     0.130891     0.135717   
 2    2  Actor_1  03-01-01-01-02-01-01.wav  neutral     0.063225     0.143786   
 3    3  Actor_1  03-01-01-01-02-02-01.wav  neutral     0.085431     0.110258   
 4    4  Actor_1  03-01-02-01-01-01-01.wav     calm     0.122966     0.138861   
 
    audio_emb_2  audio_emb_3  audio_emb_4  audio_emb_5  ...  audio_emb_54  \
 0     1.831814          0.0     0.961708          0.0  ...      1.023153   
 1     1.872676          0.0     0.876306          0.0  ...      1.004804   
 2     1.776124          0.0     0.979834          0.0  ...      1.091014   
 3     1.717838          0.0     0.883678          0.0  ...      1.244794   
 4     1.919742          0.0     0.805055          0.0  ...      0.708838   
 
    au

video

In [11]:
import os
print("AUDIO_PATH:", AUDIO_PATH)
print("VIDEO_PATH:", VIDEO_PATH)
print("META_PATH:", META_PATH)

print("\nVideo root contents:")
!ls "$VIDEO_PATH"

AUDIO_PATH: ravdess/audio
VIDEO_PATH: ravdess/video
META_PATH: ravdess/metadata

Video root contents:
Video_Speech_Actor_01 Video_Speech_Actor_03 Video_Speech_Actor_05
Video_Speech_Actor_02 Video_Speech_Actor_04 Video_Speech_Actor_06


In [12]:
import pandas as pd

# Build: video filename -> full path
video_map = {}
for root, dirs, files in os.walk(VIDEO_PATH):
    for f in files:
        if f.endswith(".mp4"):
            video_map[f] = os.path.join(root, f)

print("Number of video files found:", len(video_map))
print("Example entries:")
for k in list(video_map.keys())[:10]:
    print(" ", k, "->", video_map[k])


Number of video files found: 720
Example entries:
  02-01-08-02-02-01-02.mp4 -> ravdess/video/Video_Speech_Actor_02/02-01-08-02-02-01-02.mp4
  02-01-08-01-01-01-02.mp4 -> ravdess/video/Video_Speech_Actor_02/02-01-08-01-01-01-02.mp4
  01-01-05-01-02-02-02.mp4 -> ravdess/video/Video_Speech_Actor_02/01-01-05-01-02-02-02.mp4
  01-01-06-01-02-01-02.mp4 -> ravdess/video/Video_Speech_Actor_02/01-01-06-01-02-01-02.mp4
  01-01-06-02-01-01-02.mp4 -> ravdess/video/Video_Speech_Actor_02/01-01-06-02-01-01-02.mp4
  01-01-05-02-01-02-02.mp4 -> ravdess/video/Video_Speech_Actor_02/01-01-05-02-01-02-02.mp4
  01-01-07-01-01-02-02.mp4 -> ravdess/video/Video_Speech_Actor_02/01-01-07-01-01-02-02.mp4
  01-01-04-01-01-01-02.mp4 -> ravdess/video/Video_Speech_Actor_02/01-01-04-01-01-01-02.mp4
  01-01-04-02-02-01-02.mp4 -> ravdess/video/Video_Speech_Actor_02/01-01-04-02-02-01-02.mp4
  01-01-07-02-02-02-02.mp4 -> ravdess/video/Video_Speech_Actor_02/01-01-07-02-02-02-02.mp4


In [13]:
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

rows = []

for actor in sorted(os.listdir(AUDIO_PATH)):
    actor_dir = os.path.join(AUDIO_PATH, actor)
    if not os.path.isdir(actor_dir):
        continue

    for fname in sorted(os.listdir(actor_dir)):
        if not fname.endswith(".wav"):
            continue

        parts = fname.split("-")
        emotion_code = parts[2]
        emotion = emotion_map[emotion_code]

        # audio path relative to DATASET_PATH
        audio_rel = os.path.join("audio", actor, fname)

        # audio = 03-..., full AV = 01-..., video-only = 02-...
        parts_mp4 = parts.copy()

        parts_mp4[0] = "01"
        mp4_full = "-".join(parts_mp4).replace(".wav", ".mp4")

        parts_mp4[0] = "02"
        mp4_video_only = "-".join(parts_mp4).replace(".wav", ".mp4")

        video_abs = None
        if mp4_full in video_map:
            video_abs = video_map[mp4_full]
        elif mp4_video_only in video_map:
            video_abs = video_map[mp4_video_only]
        else:
            continue  # no matching video for this audio

        video_rel = os.path.relpath(video_abs, DATASET_PATH)

        rows.append({
            "actor": actor,
            "fname": fname,
            "audio_path": audio_rel,
            "video_path": video_rel,
            "emotion": emotion
        })

df = pd.DataFrame(rows)
os.makedirs(META_PATH, exist_ok=True)
labels_csv_path = os.path.join(META_PATH, "labels.csv")
df.to_csv(labels_csv_path, index=False)

print("Saved labels to:", labels_csv_path)
df.head()


Saved labels to: ravdess/metadata/labels.csv


,actor,fname,audio_path,video_path,emotion
0,Actor_1,03-01-01-01-01-01-01.wav,audio/Actor_1/03-01-01-01-01-01-01.wav,video/Video_Speech_Actor_01/01-01-01-01-01-01-...,neutral
1,Actor_1,03-01-01-01-01-02-01.wav,audio/Actor_1/03-01-01-01-01-02-01.wav,video/Video_Speech_Actor_01/01-01-01-01-01-02-...,neutral
2,Actor_1,03-01-01-01-02-01-01.wav,audio/Actor_1/03-01-01-01-02-01-01.wav,video/Video_Speech_Actor_01/01-01-01-01-02-01-...,neutral
3,Actor_1,03-01-01-01-02-02-01.wav,audio/Actor_1/03-01-01-01-02-02-01.wav,video/Video_Speech_Actor_01/01-01-01-01-02-02-...,neutral
4,Actor_1,03-01-02-01-01-01-01.wav,audio/Actor_1/03-01-02-01-01-01-01.wav,video/Video_Speech_Actor_01/01-01-02-01-01-01-...,calm


In [14]:
import cv2
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class RavdessVideoDataset(Dataset):
    def __init__(self, csv_path, root_dir, img_size=224):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir

        self.label2id = {label: i for i, label in enumerate(sorted(self.df["emotion"].unique()))}
        self.id2label = {v: k for k, v in self.label2id.items()}

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def _load_middle_frame(self, video_path):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {video_path}")

        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        mid_frame_idx = frame_count // 2

        cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame_idx)
        ret, frame = cap.read()
        cap.release()

        if not ret or frame is None:
            raise RuntimeError(f"Could not read frame {mid_frame_idx} from {video_path}")

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        return frame

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rel_video_path = row["video_path"]
        video_path = os.path.join(self.root_dir, rel_video_path)

        frame = self._load_middle_frame(video_path)
        img = self.transform(frame)

        label = self.label2id[row["emotion"]]
        label = torch.tensor(label, dtype=torch.long)

        return img, label


In [15]:
video_dataset = RavdessVideoDataset(labels_csv_path, DATASET_PATH)
print("Number of samples:", len(video_dataset))

sample_img, sample_label = video_dataset[0]
print("Sample image shape:", sample_img.shape)
print("Sample label:", sample_label)


Number of samples: 360
Sample image shape: torch.Size([3, 224, 224])
Sample label: tensor(5)


In [16]:
from torch.utils.data import random_split, DataLoader

dataset_size = len(video_dataset)
val_size = int(0.2 * dataset_size)
train_size = dataset_size - val_size

video_train_ds, video_val_ds = random_split(video_dataset, [train_size, val_size])

video_train_loader = DataLoader(video_train_ds, batch_size=16, shuffle=True)
video_val_loader   = DataLoader(video_val_ds, batch_size=16, shuffle=False)

len(video_train_ds), len(video_val_ds)


(288, 72)

In [17]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

num_classes = len(video_dataset.label2id)

base_model = resnet18(weights="IMAGENET1K_V1")  # pretrained
in_features = base_model.fc.in_features
base_model.fc = nn.Linear(in_features, num_classes)  # new head

video_model = base_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(video_model.parameters(), lr=1e-4)


Device: cpu


In [18]:
from tqdm.auto import tqdm

num_epochs = 10  # you can change later

for epoch in range(num_epochs):
    print(f"\n===== VIDEO Epoch {epoch+1} / {num_epochs} =====")

    # ---- Train ----
    video_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in tqdm(video_train_loader, desc=f"Train {epoch+1}", leave=False):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = video_model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ---- Validate ----
    video_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X, y in tqdm(video_val_loader, desc=f"Val {epoch+1}", leave=False):
            X, y = X.to(device), y.to(device)

            logits = video_model(X)
            loss = criterion(logits, y)

            val_loss += loss.item() * X.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")



===== VIDEO Epoch 1 / 10 =====


Train 1:   0%|          | 0/18 [00:00<?, ?it/s]

Val 1:   0%|          | 0/5 [00:00<?, ?it/s]

Train Loss: 1.8043 | Train Acc: 0.3785
Val   Loss: 1.5359 | Val   Acc: 0.4444

===== VIDEO Epoch 2 / 10 =====


Train 2:   0%|          | 0/18 [00:00<?, ?it/s]

...

Val 9:   0%|          | 0/5 [00:00<?, ?it/s]

Train Loss: 0.0176 | Train Acc: 1.0000
Val   Loss: 0.3385 | Val   Acc: 0.9167

===== VIDEO Epoch 10 / 10 =====


Train 10:   0%|          | 0/18 [00:00<?, ?it/s]

Val 10:   0%|          | 0/5 [00:00<?, ?it/s]

Train Loss: 0.0140 | Train Acc: 1.0000
Val   Loss: 0.3580 | Val   Acc: 0.9167


In [19]:
torch.save(video_model.state_dict(), "ravdess_video_resnet18.pth")
print("Saved video model.")

Saved video model.


In [20]:
import copy
import torch.nn as nn

# Make a deep copy of the trained classifier model
video_feat_model = copy.deepcopy(video_model)

# Replace the final classification layer with Identity
video_feat_model.fc = nn.Identity()

video_feat_model = video_feat_model.to(device)
video_feat_model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [21]:
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import os

feat_loader = DataLoader(video_dataset, batch_size=32, shuffle=False)

all_embeddings = []
all_meta = []

with torch.no_grad():
    for i, (X, y) in enumerate(feat_loader):
        X = X.to(device)
        feats = video_feat_model(X)  # [batch, 512]
        feats = feats.cpu().numpy()

        start = i * feat_loader.batch_size
        for j in range(feats.shape[0]):
            row = video_dataset.df.iloc[start + j]
            all_meta.append({
                "actor": row["actor"],
                "fname": row["fname"],
                "emotion": row["emotion"]
            })
        all_embeddings.append(feats)

emb_array = np.vstack(all_embeddings)
meta_df = pd.DataFrame(all_meta)
emb_cols = [f"video_emb_{k}" for k in range(emb_array.shape[1])]
emb_df = pd.DataFrame(emb_array, columns=emb_cols)

video_features_df = pd.concat([meta_df, emb_df], axis=1)
video_emb_path = os.path.join(META_PATH, "video_embeddings.csv")
video_features_df.to_csv(video_emb_path, index=False)

video_emb_path, video_features_df.head()


('ravdess/metadata/video_embeddings.csv',
      actor                     fname  emotion  video_emb_0  video_emb_1  \
 0  Actor_1  03-01-01-01-01-01-01.wav  neutral     0.584863     1.178789   
 1  Actor_1  03-01-01-01-01-02-01.wav  neutral     0.047261     1.479747   
 2  Actor_1  03-01-01-01-02-01-01.wav  neutral     0.201859     1.866628   
 3  Actor_1  03-01-01-01-02-02-01.wav  neutral     0.120563     2.474854   
 4  Actor_1  03-01-02-01-01-01-01.wav     calm     0.859557     0.030905   
 
    video_emb_2  video_emb_3  video_emb_4  video_emb_5  video_emb_6  ...  \
 0     0.532169     0.714682     0.351249     1.226052     0.132944  ...   
 1     0.308158     0.864383     0.769096     1.300040     0.066386  ...   
 2     0.233378     0.848195     0.359234     1.717651     0.103771  ...   
 3     0.702029     0.990526     0.422380     1.145453     0.003012  ...   
 4     0.085588     3.019762     1.797489     0.048414     1.150802  ...   
 
    video_emb_502  video_emb_503  video_em

In [22]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

audio_loader = DataLoader(dataset, batch_size=32, shuffle=False)

all_audio_emb = []
all_audio_meta = []

with torch.no_grad():
    for i, (X, y) in enumerate(audio_loader):
        X = X.to(device)
        # forward until before final FC
        h = model.conv(X)      # [B, 64, 1, 1]
        h = h.view(h.size(0), -1)   # [B, 64]
        feats = h.cpu().numpy()

        start = i * audio_loader.batch_size
        for j in range(feats.shape[0]):
            row = dataset.df.iloc[start + j]
            all_audio_meta.append({
                "actor": row["actor"],
                "fname": row["fname"],
                "emotion": row["emotion"]
            })
        all_audio_emb.append(feats)

audio_emb_array = np.vstack(all_audio_emb)
audio_meta_df = pd.DataFrame(all_audio_meta)
audio_emb_cols = [f"audio_emb_{k}" for k in range(audio_emb_array.shape[1])]
audio_emb_df = pd.DataFrame(audio_emb_array, columns=audio_emb_cols)

audio_features_df = pd.concat([audio_meta_df, audio_emb_df], axis=1)
audio_emb_path = os.path.join(META_PATH, "audio_embeddings.csv")
audio_features_df.to_csv(audio_emb_path, index=False)

audio_emb_path, audio_features_df.head()


('ravdess/metadata/audio_embeddings.csv',
      actor                     fname  emotion  audio_emb_0  audio_emb_1  \
 0  Actor_1  03-01-01-01-01-01-01.wav  neutral     0.091919     0.145806   
 1  Actor_1  03-01-01-01-01-02-01.wav  neutral     0.130891     0.135717   
 2  Actor_1  03-01-01-01-02-01-01.wav  neutral     0.063225     0.143786   
 3  Actor_1  03-01-01-01-02-02-01.wav  neutral     0.085431     0.110257   
 4  Actor_1  03-01-02-01-01-01-01.wav     calm     0.122965     0.138861   
 
    audio_emb_2  audio_emb_3  audio_emb_4  audio_emb_5  audio_emb_6  ...  \
 0     1.831814          0.0     0.961708          0.0     1.591410  ...   
 1     1.872676          0.0     0.876306          0.0     1.646756  ...   
 2     1.776124          0.0     0.979834          0.0     1.537120  ...   
 3     1.717838          0.0     0.883678          0.0     1.495914  ...   
 4     1.919742          0.0     0.805056          0.0     1.700731  ...   
 
    audio_emb_54  audio_emb_55  audio_emb_

In [23]:
import pandas as pd
import os

audio_df = pd.read_csv(os.path.join(META_PATH, "audio_embeddings.csv"))
video_df = pd.read_csv(os.path.join(META_PATH, "video_embeddings.csv"))

fusion_df = pd.merge(
    audio_df,
    video_df,
    on=["actor", "fname", "emotion"],
    how="inner"
)

len(fusion_df), fusion_df.head()


(360,
      actor                     fname  emotion  audio_emb_0  audio_emb_1  \
 0  Actor_1  03-01-01-01-01-01-01.wav  neutral     0.091919     0.145806   
 1  Actor_1  03-01-01-01-01-02-01.wav  neutral     0.130891     0.135717   
 2  Actor_1  03-01-01-01-02-01-01.wav  neutral     0.063225     0.143786   
 3  Actor_1  03-01-01-01-02-02-01.wav  neutral     0.085431     0.110257   
 4  Actor_1  03-01-02-01-01-01-01.wav     calm     0.122966     0.138861   
 
    audio_emb_2  audio_emb_3  audio_emb_4  audio_emb_5  audio_emb_6  ...  \
 0     1.831814          0.0     0.961708          0.0     1.591410  ...   
 1     1.872676          0.0     0.876306          0.0     1.646756  ...   
 2     1.776124          0.0     0.979834          0.0     1.537120  ...   
 3     1.717838          0.0     0.883678          0.0     1.495914  ...   
 4     1.919741          0.0     0.805056          0.0     1.700731  ...   
 
    video_emb_502  video_emb_503  video_emb_504  video_emb_505  video_emb_506 

In [24]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch

# feature columns
audio_cols = [c for c in fusion_df.columns if c.startswith("audio_emb_")]
video_cols = [c for c in fusion_df.columns if c.startswith("video_emb_")]

X_audio = fusion_df[audio_cols].values
X_video = fusion_df[video_cols].values
y_labels = fusion_df["emotion"].values

# label encode emotions
unique_emotions = sorted(fusion_df["emotion"].unique())
label2id = {e: i for i, e in enumerate(unique_emotions)}
id2label = {v: k for k, v in label2id.items()}

y = np.array([label2id[e] for e in y_labels])

# split
X_a_train, X_a_val, X_v_train, X_v_val, y_train, y_val = train_test_split(
    X_audio, X_video, y, test_size=0.2, random_state=42, stratify=y
)

class FusionDataset(Dataset):
    def __init__(self, X_a, X_v, y):
        self.X_a = torch.tensor(X_a, dtype=torch.float32)
        self.X_v = torch.tensor(X_v, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_a[idx], self.X_v[idx], self.y[idx]

fusion_train_ds = FusionDataset(X_a_train, X_v_train, y_train)
fusion_val_ds   = FusionDataset(X_a_val,   X_v_val,   y_val)

fusion_train_loader = DataLoader(fusion_train_ds, batch_size=32, shuffle=True)
fusion_val_loader   = DataLoader(fusion_val_ds,   batch_size=32, shuffle=False)

len(fusion_train_ds), len(fusion_val_ds)


(288, 72)

In [25]:
import torch.nn as nn

class FusionNet(nn.Module):
    def __init__(self, audio_dim, video_dim, num_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(audio_dim + video_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, a, v):
        x = torch.cat([a, v], dim=1)
        return self.fc(x)

audio_dim = X_audio.shape[1]
video_dim = X_video.shape[1]
num_classes = len(unique_emotions)

device = "cuda" if torch.cuda.is_available() else "cpu"
fusion_model = FusionNet(audio_dim, video_dim, num_classes).to(device)

fusion_criterion = nn.CrossEntropyLoss()
fusion_optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-3)


In [26]:
from tqdm.auto import tqdm

num_epochs = 10

for epoch in range(num_epochs):
    print(f"\n===== FUSION Epoch {epoch+1} / {num_epochs} =====")

    # Train
    fusion_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for a, v, yb in tqdm(fusion_train_loader, desc=f"Train {epoch+1}", leave=False):
        a, v, yb = a.to(device), v.to(device), yb.to(device)

        fusion_optimizer.zero_grad()
        logits = fusion_model(a, v)
        loss = fusion_criterion(logits, yb)
        loss.backward()
        fusion_optimizer.step()

        train_loss += loss.item() * a.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # Val
    fusion_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for a, v, yb in tqdm(fusion_val_loader, desc=f"Val {epoch+1}", leave=False):
            a, v, yb = a.to(device), v.to(device), yb.to(device)

            logits = fusion_model(a, v)
            loss = fusion_criterion(logits, yb)

            val_loss += loss.item() * a.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")



===== FUSION Epoch 1 / 10 =====


Train 1:   0%|          | 0/9 [00:00<?, ?it/s]

Val 1:   0%|          | 0/3 [00:00<?, ?it/s]

Train Loss: 0.9920 | Train Acc: 0.7847
Val   Loss: 0.1585 | Val   Acc: 1.0000

===== FUSION Epoch 2 / 10 =====


Train 2:   0%|          | 0/9 [00:00<?, ?it/s]

...

Val 9:   0%|          | 0/3 [00:00<?, ?it/s]

Train Loss: 0.0236 | Train Acc: 0.9931
Val   Loss: 0.0046 | Val   Acc: 1.0000

===== FUSION Epoch 10 / 10 =====


Train 10:   0%|          | 0/9 [00:00<?, ?it/s]

Val 10:   0%|          | 0/3 [00:00<?, ?it/s]

Train Loss: 0.0159 | Train Acc: 1.0000
Val   Loss: 0.0061 | Val   Acc: 1.0000


In [27]:
import pandas as pd
import torch
import torch.nn.functional as F
import numpy as np
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
video_model.to(device)
fusion_model.to(device)

model.eval()
video_model.eval()
fusion_model.eval()

# Make sure we know which columns are embeddings
audio_cols = [c for c in fusion_df.columns if c.startswith("audio_emb_")]
video_cols = [c for c in fusion_df.columns if c.startswith("video_emb_")]

# Build lookup: (actor, fname) -> index in audio/video datasets
audio_index_map = {
    (row["actor"], row["fname"]): idx
    for idx, row in dataset.df.reset_index().iterrows()
}

video_index_map = {
    (row["actor"], row["fname"]): idx
    for idx, row in video_dataset.df.reset_index().iterrows()
}


In [28]:
import numpy as np

audio_cols = [c for c in fusion_df.columns if c.startswith("audio_emb_")]
video_cols = [c for c in fusion_df.columns if c.startswith("video_emb_")]

fusion_df[audio_cols] = fusion_df[audio_cols].apply(
    pd.to_numeric, errors="coerce"
)
fusion_df[video_cols] = fusion_df[video_cols].apply(
    pd.to_numeric, errors="coerce"
)

fusion_df[audio_cols].dtypes.head(), fusion_df[video_cols].dtypes.head()


(audio_emb_0    float64
 audio_emb_1    float64
 audio_emb_2    float64
 audio_emb_3    float64
 audio_emb_4    float64
 dtype: object,
 video_emb_0    float64
 video_emb_1    float64
 video_emb_2    float64
 video_emb_3    float64
 video_emb_4    float64
 dtype: object)

In [29]:
rows = []

with torch.no_grad():
    for i, row in fusion_df.iterrows():
        actor = row["actor"]
        fname = row["fname"]
        true_emotion = row["emotion"]

        key = (actor, fname)

        # --- AUDIO prediction from raw audio model ---
        a_idx = audio_index_map[key]
        x_a, _ = dataset[a_idx]
        x_a = x_a.unsqueeze(0).to(device)

        logits_a = model(x_a)
        probs_a = F.softmax(logits_a, dim=1).cpu().numpy()[0]
        audio_pred_id = int(np.argmax(probs_a))
        audio_pred = dataset.id2label[audio_pred_id]
        audio_conf = float(np.max(probs_a))

        # --- VIDEO prediction from raw video model ---
        v_idx = video_index_map[key]
        x_v, _ = video_dataset[v_idx]
        x_v = x_v.unsqueeze(0).to(device)

        logits_v = video_model(x_v)
        probs_v = F.softmax(logits_v, dim=1).cpu().numpy()[0]
        video_pred_id = int(np.argmax(probs_v))
        video_pred = video_dataset.id2label[video_pred_id]
        video_conf = float(np.max(probs_v))

        # --- FUSION prediction from FusionNet using embeddings ---
        # Force-cast the row slice to float32 NumPy before making a tensor
        a_vals = row[audio_cols].to_numpy(dtype=np.float32)
        v_vals = row[video_cols].to_numpy(dtype=np.float32)

        a_emb = torch.from_numpy(a_vals).unsqueeze(0).to(device)  # [1, audio_dim]
        v_emb = torch.from_numpy(v_vals).unsqueeze(0).to(device)  # [1, video_dim]

        logits_f = fusion_model(a_emb, v_emb)
        probs_f = F.softmax(logits_f, dim=1).cpu().numpy()[0]
        fusion_id = int(np.argmax(probs_f))
        fusion_pred = id2label[fusion_id]
        fusion_conf = float(np.max(probs_f))

        rows.append({
            "actor": actor,
            "fname": fname,
            "true_emotion": true_emotion,
            "audio_pred": audio_pred,
            "audio_conf": audio_conf,
            "video_pred": video_pred,
            "video_conf": video_conf,
            "fusion_pred": fusion_pred,
            "fusion_conf": fusion_conf
        })

pred_df = pd.DataFrame(rows)

pred_csv_path = os.path.join(META_PATH, "multimodal_predictions.csv")
pred_df.to_csv(pred_csv_path, index=False)

pred_csv_path, pred_df.head()


('ravdess/metadata/multimodal_predictions.csv',
      actor                     fname true_emotion audio_pred  audio_conf  \
 0  Actor_1  03-01-01-01-01-01-01.wav      neutral  surprised    0.477590   
 1  Actor_1  03-01-01-01-01-02-01.wav      neutral  surprised    0.390899   
 2  Actor_1  03-01-01-01-02-01-01.wav      neutral  surprised    0.520646   
 3  Actor_1  03-01-01-01-02-02-01.wav      neutral  surprised    0.487337   
 4  Actor_1  03-01-02-01-01-01-01.wav         calm  surprised    0.264630   
 
   video_pred  video_conf fusion_pred  fusion_conf  
 0    neutral    0.996069     neutral     0.998140  
 1    neutral    0.996634     neutral     0.999168  
 2    neutral    0.998871     neutral     0.999553  
 3    neutral    0.998836     neutral     0.999379  
 4       calm    0.998881        calm     0.999762  )

In [30]:
import pandas as pd
import os

pred_csv_path = os.path.join(META_PATH, "multimodal_predictions.csv")
pred_df = pd.read_csv(pred_csv_path)

pred_df["audio_correct"] = (pred_df["audio_pred"] == pred_df["true_emotion"]).astype(int)
pred_df["video_correct"] = (pred_df["video_pred"] == pred_df["true_emotion"]).astype(int)
pred_df["fusion_correct"] = (pred_df["fusion_pred"] == pred_df["true_emotion"]).astype(int)

# overall accuracy per modality (just to print)
print("Audio acc: ", pred_df["audio_correct"].mean())
print("Video acc: ", pred_df["video_correct"].mean())
print("Fusion acc:", pred_df["fusion_correct"].mean())

final_csv_path = os.path.join(META_PATH, "multimodal_predictions_with_flags.csv")
pred_df.to_csv(final_csv_path, index=False)

final_csv_path, pred_df.head()


Audio acc:  0.35555555555555557
Video acc:  0.9833333333333333
Fusion acc: 1.0


('ravdess/metadata/multimodal_predictions_with_flags.csv',
      actor                     fname true_emotion audio_pred  audio_conf  \
 0  Actor_1  03-01-01-01-01-01-01.wav      neutral  surprised    0.477590   
 1  Actor_1  03-01-01-01-01-02-01.wav      neutral  surprised    0.390899   
 2  Actor_1  03-01-01-01-02-01-01.wav      neutral  surprised    0.520646   
 3  Actor_1  03-01-01-01-02-02-01.wav      neutral  surprised    0.487337   
 4  Actor_1  03-01-02-01-01-01-01.wav         calm  surprised    0.264630   
 
   video_pred  video_conf fusion_pred  fusion_conf  audio_correct  \
 0    neutral    0.996069     neutral     0.998140              0   
 1    neutral    0.996634     neutral     0.999168              0   
 2    neutral    0.998871     neutral     0.999553              0   
 3    neutral    0.998836     neutral     0.999379              0   
 4       calm    0.998881        calm     0.999762              0   
 
    video_correct  fusion_correct  
 0              1        